# 04 - Exploratory Data Analysis

look at the feature table before training anything: missingness, distributions, correlations, and how the two labels break down by chronic condition. this runs *after* feature engineering (03) and *before* modeling (05) on purpose -- everything here is about understanding `features_labeled.parquet`, not building it.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_parquet("../data/processed/features_labeled.parquet")
print(f"{len(df):,} members, {df.shape[1]} columns")
df.head()


**missing values.** notebook 03 already fills any unmatched member with 0 before saving (e.g. someone with zero carrier claims in the window has `carrier_claims = 0`, not null), so there shouldn't be any nulls left here -- this just confirms that.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0]
print("columns with missing values:" if len(missing) else "no missing values in any column")
missing


**descriptive stats.** healthcare cost/utilization data is almost always heavily right-skewed -- most members cost little, a small tail costs a lot -- so the mean is a misleading summary on its own. the 95th/99th percentiles below make that skew visible directly.

In [ ]:
spend_cols = ["inpatient_paid", "outpatient_paid", "carrier_paid", "rx_total_cost", "other_payer_paid", "total_paid_2010"]
df[spend_cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T


In [ ]:
util_cols = ["inpatient_admits", "outpatient_visits", "carrier_claims", "rx_fill_count", "inpatient_admits_2010"]
df[util_cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T


**label balance.** both labels are rare by construction -- `high_cost` is pinned to the 90th percentile, `high_utilization` to 2+ admits -- so this is really just a visual confirmation of numbers already seen in notebook 03.\n\n**a note on which `high_cost` this is:** the `high_cost` column here is a *population-level* definition -- the 90th percentile of the full 52,375-member cohort's 2010 spend, fine for a descriptive/EDA view of the population. The actual modeling notebooks (05, 08) don't use this column to train or evaluate -- they derive their own threshold from the training split only, to avoid the test set's outcome distribution influencing the label boundary. The two thresholds are very close numerically, but they're not the same object -- see notebook 05 for why.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, label in zip(axes, ["high_cost", "high_utilization"]):
    counts = df[label].value_counts().sort_index()
    ax.bar(["negative", "positive"], counts.values, color=["#4c5a57", "#1f6f68"])
    ax.set_title(f"{label} ({counts[1] / counts.sum():.1%} positive)")
    for i, v in enumerate(counts.values):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()


**distributions of the continuous features.** spend and rx cost are shown on a log1p scale -- on a raw scale the right tail flattens everything else in the histogram into an unreadable spike near zero.

In [ ]:
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
plot_cols = ["age", "inpatient_admits", "outpatient_visits", "carrier_paid", "rx_total_cost", "other_payer_paid"]
for ax, col in zip(axes.flat, plot_cols):
    values = df[col]
    if values.max() > 50:  # skewed count/dollar columns get a log1p view
        ax.hist(np.log1p(values), bins=40, color="#1f6f68")
        ax.set_xlabel(f"log1p({col})")
    else:
        ax.hist(values, bins=40, color="#1f6f68")
        ax.set_xlabel(col)
    ax.set_ylabel("members")
plt.tight_layout()
plt.show()


**correlation among features.** mainly a multicollinearity check before modeling -- e.g. `inpatient_paid` and `inpatient_admits` are almost definitionally correlated (more admits, more paid), which matters for interpreting logistic regression coefficients even though it doesn't hurt xgboost.

In [ ]:
numeric_cols = [c for c in df.columns if c not in ("desynpuf_id",) and df[c].dtype != "object"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="RdBu_r", center=0, square=True, linewidths=0.3, ax=ax,
            cbar_kws={"shrink": 0.7})
ax.set_title("feature correlation matrix")
plt.tight_layout()
plt.show()


**what correlates with each label.** a quick point-biserial correlation (pandas treats the 0/1 label as numeric, which is exactly a point-biserial correlation) -- a cheap first look at what's likely to matter before running shap in notebook 06.

In [ ]:
feature_cols = [c for c in numeric_cols if c not in ("high_cost", "high_utilization", "total_paid_2010", "inpatient_admits_2010")]

corr_with_labels = pd.DataFrame({
    "corr_with_high_cost": df[feature_cols + ["high_cost"]].corr()["high_cost"].drop("high_cost"),
    "corr_with_high_utilization": df[feature_cols + ["high_utilization"]].corr()["high_utilization"].drop("high_utilization"),
}).sort_values("corr_with_high_cost", ascending=False)

corr_with_labels


**chronic condition prevalence by cost status.** the number a care management team actually wants: among members who turned out high-cost, what share have each condition, versus the rest of the cohort?

In [ ]:
condition_cols = ["has_chf", "has_diabetes", "has_ckd", "has_copd", "has_cancer",
                   "has_alzheimers", "has_depression", "has_ihd", "has_osteoporosis",
                   "has_ra_oa", "has_stroke_tia", "has_esrd"]

prevalence = df.groupby("high_cost")[condition_cols].mean().T
prevalence.columns = ["not_high_cost", "high_cost"]
prevalence["lift"] = prevalence["high_cost"] / prevalence["not_high_cost"]
prevalence.sort_values("lift", ascending=False)


**data quality checks.** a few things worth confirming explicitly rather than assuming -- one row per member, sane age range, mutually exclusive race dummies, and how many rows have a negative paid amount (a real de-synpuf quirk from claim adjustments/reversals, not a pipeline bug -- just worth knowing the count).

In [ ]:
print("duplicate desynpuf_id rows:", df["desynpuf_id"].duplicated().sum())
print(f"age range: {df['age'].min()}-{df['age'].max()}")
print("age outside 18-110:", ((df['age'] < 18) | (df['age'] > 110)).sum())

race_sum = df["race_black"] + df["race_other"] + df["race_hispanic"]
print("race one-hot values (should only ever be 0 or 1):", sorted(race_sum.unique()))

print("rows with negative total_paid_2010:", (df["total_paid_2010"] < 0).sum())
print("rows with negative inpatient_paid:", (df["inpatient_paid"] < 0).sum())


**multicollinearity.** pairs above |r| = 0.5. most of these are near-definitional -- a claim-count and its own dollar amount are always going to move together -- and don't hurt xgboost, but they do make individual logistic regression coefficients hard to interpret in isolation (the model can't tell whether `carrier_claims` or `carrier_paid` is doing the work when the two move together 94% of the time).

In [ ]:
import numpy as np

corr_matrix = df[numeric_cols].corr()
pairs = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)).stack()
high_corr = pairs[pairs.abs() > 0.5].sort_values(key=abs, ascending=False)
high_corr


**a guardrail worth repeating out loud**: `total_paid_2010` and `inpatient_admits_2010` sit in this same table and correlate highly with the labels *because they define the labels* -- that's expected, not leakage. but it's exactly the kind of column a future change could accidentally add to `FEATURE_COLS` in notebook 05 and silently turn into real leakage. they should never appear there.